# Activity 5: Building the dataset

We put Trinidad and Tobago's quarterly real GDP together with three high-frequency
sources from the earlier activities and the oil price:

| Source | File | Frequency |
|---|---|---|
| Real GDP (CBTT, constant prices) | `data/gdp.csv` | quarterly |
| Google Trends (25 search terms) | `data/gtrends.csv` | monthly |
| Night-time lights, country mean | `data/ntl_shape.csv` | daily |
| Brent oil price (EIA) | `data/eia_brent.csv` | daily |

Steps: load → clean → seasonally adjust **each series at its own frequency** → merge to
quarterly → growth rates → a first OLS. Then the question that matters: does the
relationship survive without 2020?

Kernel: `nwcst`. Cells marked `# YOUR CODE HERE` are yours.

In [ ]:
# ===========================================================================
# CONFIG LOADER  --  edit config.py, NOT this cell.
# Gives you `cfg` (every setting) and `P` (the standard folders).
# ===========================================================================
import os, sys, importlib.util

# config.py lives in activity/. From activity/ that is the current folder;
# from sols/ it is next door.
CODE_DIR = os.getcwd()
if not os.path.exists(os.path.join(CODE_DIR, "config.py")):
    CODE_DIR = os.path.abspath(os.path.join(CODE_DIR, "..", "activity"))
sys.path.insert(0, CODE_DIR)       # so `import nwcst_helpers` works from sols/ too

_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)     # raw/, data/, output/ under public/

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE})  data -> {P['data']}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

## 1. GDP

`gdp.csv` stamps each quarter on its **last** month (2026Q1 = `2026-03-01`).

In [ ]:
gdp = pd.read_csv(os.path.join(P["data"], cfg.GDP_CSV), index_col="date", parse_dates=True)
gdp = gdp.rename(columns={"rgdp": "RGDP0000"})
print(gdp.index.min().date(), "->", gdp.index.max().date(), f"({len(gdp)} quarters)")
gdp.tail()

In [ ]:
ax = gdp["RGDP0000"].plot(figsize=(9, 3), title="Trinidad and Tobago: real GDP (TT$ mn, constant prices)")
ax.set_xlabel("");

GDP is **not** seasonally adjusted. With quarterly data one seasonal cycle is **4**
observations, so `period=4`.

In [ ]:
seasonal_decompose(gdp["RGDP0000"], model="additive", extrapolate_trend="freq", period=4).plot();

In [ ]:
gdp["RGDP0000"] = gdp["RGDP0000"] - seasonal_decompose(
    gdp["RGDP0000"], model="additive", extrapolate_trend="freq", period=4).seasonal

## 2. Google Trends

A zero in Google Trends usually means *too few searches to report*, not *no interest*, so
we treat zeros as missing, drop terms that are mostly missing, and average the rest into
one index.

In [ ]:
gt = pd.read_csv(os.path.join(P["data"], cfg.GTRENDS_CSV), index_col="date", parse_dates=True)
gt = gt.loc["2012-01-01":]                       # same window as the night lights
gt = gt.replace(0, np.nan)
gt = gt.dropna(axis=1, thresh=int(0.75 * len(gt)))   # keep terms with >= 75% of months
print(gt.shape[1], "terms kept:", ", ".join(c.replace("gtrends_", "") for c in gt.columns))
gt["gt_mean"] = gt.mean(axis=1)
gt["gt_mean"].plot(figsize=(9, 3), title="Google Trends: average of the kept terms");

**Exercise 1.** Seasonally adjust `gt_mean` (monthly, so `period=12`) and plot the raw and adjusted series together.

In [ ]:
# YOUR CODE HERE: create gt["gt_mean_sa"] = gt_mean minus its seasonal component, then plot both

In [ ]:
gt = gt[["gt_mean_sa"]].rename(columns={"gt_mean_sa": "gt_mean"})

## 3. Night-time lights

The country-mean radiance from Activity 3 is **daily**. Cloudy nights leave gaps, so we
average to months first.

In [ ]:
ntl = pd.read_csv(os.path.join(P["data"], cfg.NTL_CSV), index_col="date", parse_dates=True)
ntl = ntl[["shape_mean"]].resample("MS").mean().rename(columns={"shape_mean": "ntl_mean"})
ntl = ntl.loc[:"2026-09-01"]                     # the current month is incomplete
ntl.plot(figsize=(9, 3), title="Night-time lights: monthly mean radiance (nW/cm²/sr)");

**Exercise 2.** Seasonally adjust `ntl_mean` exactly as you did for Google Trends.

In [ ]:
# YOUR CODE HERE: ntl["ntl_mean_sa"] = ...

In [ ]:
ntl = ntl[["ntl_mean_sa"]].rename(columns={"ntl_mean_sa": "ntl_mean"})

## 4. Brent oil price

Trinidad and Tobago is an energy exporter, so the oil price is a natural candidate. The EIA
series in `data/eia_brent.csv` is **daily**; we average it to months. Prices are not
seasonally adjusted.

In [ ]:
brent = pd.read_csv(os.path.join(P["data"], "eia_brent.csv"), index_col="date", parse_dates=True)
brent = brent.rename(columns={"brent price": "brent"}).resample("MS").mean().loc["2012":]
brent.plot(figsize=(9, 3), title="Brent, US$ per barrel (monthly mean)");

## 5. Merge to quarterly

Monthly series become quarterly by averaging the three months. `resample("QS")` labels a
quarter by its first month, so we shift GDP's stamp (last month) to the same label first.

In [ ]:
gdp_q = gdp.copy()
gdp_q.index = gdp_q.index.to_period("Q").to_timestamp()      # 2026-03-01 -> 2026-01-01

monthly = pd.concat([gt, ntl, brent], axis=1)

**Exercise 3.** Resample `monthly` to quarters (`"QS"`) using the mean, then join it with `gdp_q`.

In [ ]:
# YOUR CODE HERE: quarterly = ...
#                 data = pd.concat([gdp_q, quarterly], axis=1)

In [ ]:
data = data.loc[:gdp_q.index.max()]          # stop at the last published GDP quarter
data.tail(8)

## 6. Growth rates

Quarter-on-quarter growth, in percent.

In [ ]:
growth = data.pct_change(fill_method=None).mul(100)
growth.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for col, color in [("RGDP0000", "#003865"), ("gt_mean", "#009CA7"), ("ntl_mean", "#F5A623")]:
    ax.plot(growth[col], label=col, color=color, lw=2 if col == "RGDP0000" else 1.2)
ax.axhline(0, color="black", lw=0.5)
ax.set_title("Quarterly growth, % q/q (seasonally adjusted)")
ax.legend(frameon=False);

In [ ]:
growth.corr().round(2)

## 7. A first OLS

$$\text{GDP growth}_t = \beta_0 + \beta_1\,\text{Google Trends}_t + \beta_2\,\text{NTL}_t + \varepsilon_t$$

In [ ]:
import statsmodels.api as sm

d1 = growth[["RGDP0000", "gt_mean", "ntl_mean"]].dropna()
m1 = sm.OLS(d1["RGDP0000"], sm.add_constant(d1[["gt_mean", "ntl_mean"]])).fit()
print(f"{len(d1)} quarters, {d1.index.min().date()} -> {d1.index.max().date()}")
print(m1.summary())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(d1.index, d1["RGDP0000"], "o-", color="#003865", label="actual")
ax.plot(d1.index, m1.fittedvalues, "--", color="#F5A623", label="fitted")
ax.axhline(0, color="black", lw=0.5)
ax.set_title(f"GDP growth, actual vs fitted (R² = {m1.rsquared:.2f})")
ax.legend(frameon=False);

## 8. The question that matters: drop 2020

COVID-19 produced the largest quarterly swings in the sample. A regression can look
good simply because two or three quarters line up.

**Exercise 4.** Re-estimate the model without 2020 and compare the coefficients and R².

In [ ]:
# YOUR CODE HERE: d2 = d1[d1.index.year != 2020]; m2 = sm.OLS(...).fit()

**Exercise 5.** Add `brent` to the regression (with and without 2020). Does the oil price
help more than the searches and the lights in an energy-heavy economy? Why might the
*same-quarter* oil price be a weak predictor of real (constant-price) GDP?

In [ ]:
# YOUR CODE HERE

## Takeaways

- Seasonally adjust each series **at its own frequency** (12 for monthly, 4 for quarterly)
  *before* converting frequencies.
- One or two indicators rarely explain GDP growth in a commodity economy; the nowcast
  tool (Activities 10–11) chooses among ~500 series instead.
- Always check whether a relationship survives without the outlier quarters.